# Projet 6 - Initiez-vous au MLOps (partie 1/2)


**Objectif du projet**  
Prédire la probabilité de défaut de paiement d'un client (TARGET = 1)  
→ Problème de **classification binaire déséquilibrée**

**Approche MLOps visée dans cette première partie**  
- Suivi systématique des expériences avec **MLflow**  
- Comparaison de plusieurs modèles / jeux de features  
- Versionning des modèles  



Date : Janvier 2026  
Auteur : Joannes Landy

## Étape 2 - Traquez les expérimentations avec MLFlow

Objectifs:

Des runs visibles dans l’UI MLflow avec les paramètres testés et les scores obtenus.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import mlflow
import os


# configuration des paramètres de connexion et des répertoires de travail
MLFLOW_URL = "http://localhost:5000"
DATA_DIR = str(Path.home() / "data")
EXPERIMENT_NAME = "Projet 06 - OpenClassrooms v4"


In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier


from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, make_scorer, accuracy_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.metrics import confusion_matrix, classification_report


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.pipeline import Pipeline


import time


output_file = DATA_DIR + "/processed/application_train_processed.parquet"
df_prepossessed = pd.read_parquet(output_file)
    
# Création du jeu de test/train
random_state = 42
test_size = 0.2
target = 'TARGET'

# Split X and y
X = df_prepossessed.drop(columns=[target])
y = df_prepossessed[target]


display("X shape;", X.shape)


In [3]:

# -----------------------------
# 2. Identifier colonnes numériques et catégorielles
# -----------------------------
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Features numériques : {numeric_features}")
print(f"Features catégorielles : {categorical_features}")

# -----------------------------
# 3. Créer le préprocesseur
# -----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
    ]
)

# -----------------------------
# 4. Créer le pipeline complet
# -----------------------------
model = DummyClassifier()

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", model)
])

# -----------------------------
# 5. Séparer train / test
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    stratify=y,          # classification équilibrée
    random_state=random_state
)

# -----------------------------
# 6. Entraîner le pipeline
# -----------------------------
pipeline.fit(X_train, y_train)

# -----------------------------
# 7. Évaluer
# -----------------------------
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Précision sur le test : {accuracy:.4f}")
print("\nRapport de classification :\n")
print(classification_report(y_test, y_pred))


    

Features numériques : ['SK_ID_CURR', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'YEARS_BEGINEXPLUATATION_AVG', 'FLOORSMAX_AVG', 'YEARS_BEGINEXPLUATATION_MODE', 'FLOORSMAX_MODE', 'YEARS_BEGINEXPLUATATION_MEDI', 'FLOORSMAX_MEDI', 'TOTALAREA_MODE', 'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE', 'DAYS_LAST_PHONE_CHANGE', 'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_

/home/joannes/Documents/openclassroom/06_Initiez_vous_au_MLOps_1-2/openclassrooms_projet06/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/joannes/Documents/openclassroom/06_Initiez_vous_au_MLOps_1-2/openclassrooms_projet06/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/joannes/Documents/openclassroom/06_Initiez_vous_au_MLOps_1-2/openclassrooms_projet06/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarn

In [4]:
# Liste des modèles pour itération
models = {
    'dummy_model': DummyClassifier(),
    'linear_model': LogisticRegression(),
    'rf_model': RandomForestClassifier(),

}

# Liste des parametres à tester
param_grids = {
    'dummy_model': {},
    'linear_model': {
        'random_state': [random_state],
        'max_iter': [100, 1000],
        'class_weight': ['balanced'],
    },
    'rf_model': {
        'random_state': [random_state],
        'class_weight': ['balanced'],
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7],
        #'min_samples_split': [2, 5, 10],
        #'min_samples_leaf': [1, 2, 4]
    },


}

# reseau de neurone : MLP, plsuieur package a explorer, pytorch,

# chercher et Stocker les meilleurs modèles

def search_bestmodel(models, param_grids, X, y):
    """ Use GridSearchCV to find best parameter, return the best model"""
    best_models = {}
    for name, model in models.items():
        print(f"\nEntraînement de {name}...")
        
        numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
        categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


        preprocessor = ColumnTransformer(transformers=[
                ("num", StandardScaler(), numeric_features),
                ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
                ])
        pipeline = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("classifier", model)
                ])
        
        # GridSearchCV
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grids[name],
            scoring='f1',
            cv=5,                  # validation croisée à 5 folds
            n_jobs=-1,             # utiliser tous les cœurs
            verbose=1              # afficher la progression
        )
        
        # Ajuster sur les données d'entraînement
        grid.fit(X, y)
        
        # Sauvegarder le meilleur modèle
        best_models[name] = grid.best_estimator_
        
        print(f"Meilleurs paramètres : {grid.best_params_}")
        print(f"Meilleur score F1 : {grid.best_score_:.4f}")
    
    return best_models

best_models = search_bestmodel(models, param_grids, X, y)


Entraînement de dummy_model...
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Meilleurs paramètres : {}
Meilleur score F1 : 0.0000

Entraînement de linear_model...
Fitting 5 folds for each of 2 candidates, totalling 10 fits


ValueError: Invalid parameter 'class_weight' for estimator Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['SK_ID_CURR', 'CNT_CHILDREN',
                                                   'AMT_INCOME_TOTAL',
                                                   'AMT_CREDIT', 'AMT_ANNUITY',
                                                   'AMT_GOODS_PRICE',
                                                   'REGION_POPULATION_RELATIVE',
                                                   'DAYS_BIRTH',
                                                   'DAYS_EMPLOYED',
                                                   'DAYS_REGISTRATION',
                                                   'DAYS_ID_PUBLISH',
                                                   'FLAG_MOBIL',
                                                   'FLAG_EMP_PHONE',
                                         